# Phase 7 — Dashboard Data Preparation

## Business purpose

Phase 7 turns the Rail Operations Forecaster into an interactive decision-support dashboard. Before building the Streamlit app, this notebook creates a clean dashboard-ready dataset that contains the core outputs from the forecasting, breach-detection, and decision-support layers.

The dashboard should not retrain models every time it opens. Instead, this notebook prepares a reusable CSV that the app can read quickly.

## Plain-English logic

This notebook rebuilds the core project logic in one place:

- load the synthetic terminal dwell dataset
- train the tuned LightGBM regression model
- generate predicted dwell
- assign operational risk tiers
- train the dedicated breach classifier
- generate breach probabilities and warning flags
- save a clean dashboard-ready output file

## Output

This notebook creates:

`data/processed/dashboard_forecasts.csv`

That file will be used by the Phase 7 Streamlit dashboard.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import roc_auc_score, average_precision_score

pd.set_option("display.max_columns", None)
np.random.seed(42)

DATA_PATH = Path("../data/synthetic/phase1_terminal_dwell.csv")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_PATH = OUTPUT_DIR / "dashboard_forecasts.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("Output path:", OUTPUT_PATH)

Data path: ../data/synthetic/phase1_terminal_dwell.csv
Output path: ../data/processed/dashboard_forecasts.csv


## 1. Load the synthetic dataset

The dashboard data starts from the same synthetic terminal dwell dataset used throughout the project. The data is sorted by terminal and date to stay consistent with the earlier notebooks.

In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values(["terminal_id", "date"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Terminals:", df["terminal_id"].nunique())

df.head()

Dataset shape: (8760, 13)
Date range: 2022-01-01 00:00:00 to 2024-12-30 00:00:00
Terminals: 8


,date,terminal_id,terminal_name,region,inbound_train_count,inbound_car_count,cars_on_hand,yard_occupancy_pct,crew_starts_available,locomotive_availability_pct,is_weekend,month,target_dwell_hours
0,2022-01-01,T01,Barstow,West,13,304,532,59.1,13,92.3,1,1,19.7
1,2022-01-02,T01,Barstow,West,11,223,461,51.2,11,95.3,1,1,19.5
2,2022-01-03,T01,Barstow,West,10,219,448,49.8,18,85.7,0,1,20.0
3,2022-01-04,T01,Barstow,West,7,138,350,38.9,14,88.5,0,1,20.4
4,2022-01-05,T01,Barstow,West,9,175,361,40.1,17,90.5,0,1,20.7


## 2. Define the official feature set

This notebook uses the same official feature set established in Phase 1. Keeping the same target, features, and split date ensures that the dashboard data stays aligned with the validated project workflow.

The target is:

`target_dwell_hours`

The split date is:

`2024-07-01`

Rows before the split are used for training. Rows on or after the split are used for the dashboard output.

In [3]:
target_col = "target_dwell_hours"

feature_cols = [
    "terminal_id",
    "inbound_train_count",
    "inbound_car_count",
    "cars_on_hand",
    "yard_occupancy_pct",
    "crew_starts_available",
    "locomotive_availability_pct",
    "is_weekend",
    "month",
]

split_date = "2024-07-01"

train_df = df[df["date"] < split_date].copy()
test_df = df[df["date"] >= split_date].copy()

train_df["terminal_id"] = train_df["terminal_id"].astype("category")
test_df["terminal_id"] = test_df["terminal_id"].astype("category")

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (7296, 13)
Test shape: (1464, 13)


## 3. Train the tuned LightGBM regression model

The tuned LightGBM regression model remains the main forecasting engine for predicted dwell. This model supports planning by estimating the expected dwell magnitude for each terminal-day.

In [4]:
reg_model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=200,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=20,
    random_state=42
)

reg_model.fit(
    X_train,
    y_train,
    categorical_feature=["terminal_id"]
)

test_pred = reg_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, test_pred))
mae = mean_absolute_error(y_test, test_pred)

print(f"Regression RMSE: {rmse:.3f}")
print(f"Regression MAE: {mae:.3f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000390 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1066
[LightGBM] [Info] Number of data points in the train set: 7296, number of used features: 9
[LightGBM] [Info] Start training from score 19.060458
Regression RMSE: 3.749
Regression MAE: 2.879


## 4. Create operational risk tiers

The dashboard needs clear operational labels, not just predicted dwell values. This section converts predicted dwell into the Phase 3 risk tiers:

- Normal
- Elevated Risk
- High Risk
- Threshold Breach Warning

These tiers help users scan the dashboard quickly and identify which terminal-days need attention.

In [5]:
def assign_risk_tier(predicted_dwell):
    if predicted_dwell >= 28:
        return "Threshold Breach Warning"
    elif predicted_dwell >= 24:
        return "High Risk"
    elif predicted_dwell >= 20:
        return "Elevated Risk"
    else:
        return "Normal"


dashboard_df = test_df.copy()
dashboard_df["predicted_dwell"] = test_pred
dashboard_df["prediction_error"] = dashboard_df["predicted_dwell"] - dashboard_df[target_col]
dashboard_df["absolute_error"] = dashboard_df["prediction_error"].abs()

dashboard_df["risk_tier"] = dashboard_df["predicted_dwell"].apply(assign_risk_tier)

dashboard_df["actual_breach_24h"] = (dashboard_df[target_col] >= 24).astype(int)
dashboard_df["regression_breach_flag"] = (dashboard_df["predicted_dwell"] >= 24).astype(int)

dashboard_df[[
    "date",
    "terminal_id",
    "target_dwell_hours",
    "predicted_dwell",
    "risk_tier",
    "actual_breach_24h",
    "regression_breach_flag"
]].head()

,date,terminal_id,target_dwell_hours,predicted_dwell,risk_tier,actual_breach_24h,regression_breach_flag
912,2024-07-01,T01,18.2,21.174817,Elevated Risk,0,0
913,2024-07-02,T01,16.5,20.234585,Elevated Risk,0,0
914,2024-07-03,T01,20.4,20.413917,Elevated Risk,0,0
915,2024-07-04,T01,17.7,20.555955,Elevated Risk,0,0
916,2024-07-05,T01,18.4,20.759380,Elevated Risk,0,0


## 5. Add the resource pressure flag

The resource pressure flag follows the Phase 3 decision-support logic. It identifies terminal-days where high predicted dwell coincides with operational pressure.

The flag fires when:

- the row is High Risk or Threshold Breach Warning, and
- yard occupancy is above 75%, or crew starts available are below 10

This keeps the flag tied to both forecast risk and operational stress.

In [6]:
dashboard_df["resource_pressure_flag"] = (
    dashboard_df["risk_tier"].isin(["High Risk", "Threshold Breach Warning"]) &
    (
        (dashboard_df["yard_occupancy_pct"] > 75) |
        (dashboard_df["crew_starts_available"] < 10)
    )
).astype(int)

dashboard_df[[
    "date",
    "terminal_id",
    "predicted_dwell",
    "risk_tier",
    "yard_occupancy_pct",
    "crew_starts_available",
    "resource_pressure_flag"
]].head()

,date,terminal_id,predicted_dwell,risk_tier,yard_occupancy_pct,crew_starts_available,resource_pressure_flag
912,2024-07-01,T01,21.174817,Elevated Risk,10.0,22,0
913,2024-07-02,T01,20.234585,Elevated Risk,25.1,22,0
914,2024-07-03,T01,20.413917,Elevated Risk,12.1,20,0
915,2024-07-04,T01,20.555955,Elevated Risk,10.0,15,0
916,2024-07-05,T01,20.759380,Elevated Risk,10.0,15,0


## 6. Train the dedicated breach classifier

The regression model is useful for forecasting dwell magnitude, but Phase 5 showed that a dedicated classifier is much better for detecting 24-hour breach events.

This section trains the LightGBM classifier using the same feature set and adds breach probabilities to the dashboard data.

In [7]:
breach_threshold = 24.0

train_b = train_df.copy()
test_b = test_df.copy()

train_b["breach_24h"] = (train_b[target_col] >= breach_threshold).astype(int)
test_b["breach_24h"] = (test_b[target_col] >= breach_threshold).astype(int)

X_train_b = train_b[feature_cols].copy()
y_train_b = train_b["breach_24h"].copy()

X_test_b = test_b[feature_cols].copy()
y_test_b = test_b["breach_24h"].copy()

X_train_b["terminal_id"] = X_train_b["terminal_id"].astype("category")
X_test_b["terminal_id"] = X_test_b["terminal_id"].astype("category")

neg_count = (y_train_b == 0).sum()
pos_count = (y_train_b == 1).sum()
scale_pos_weight = neg_count / pos_count

print("Negative class count:", neg_count)
print("Positive class count:", pos_count)
print("scale_pos_weight:", round(scale_pos_weight, 3))

Negative class count: 6248
Positive class count: 1048
scale_pos_weight: 5.962


In [8]:
clf_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=20,
    min_child_samples=20,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

clf_model.fit(
    X_train_b,
    y_train_b,
    categorical_feature=["terminal_id"]
)

breach_prob = clf_model.predict_proba(X_test_b)[:, 1]

roc_auc = roc_auc_score(y_test_b, breach_prob)
avg_precision = average_precision_score(y_test_b, breach_prob)

print(f"Classifier ROC-AUC: {roc_auc:.3f}")
print(f"Classifier Average Precision: {avg_precision:.3f}")

[LightGBM] [Info] Number of positive: 1048, number of negative: 6248
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000156 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1066
[LightGBM] [Info] Number of data points in the train set: 7296, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.143640 -> initscore=-1.785378
[LightGBM] [Info] Start training from score -1.785378
Classifier ROC-AUC: 0.757
Classifier Average Precision: 0.351


## 7. Add breach probabilities and warning flags

The dashboard should show both:

- the regression-based dwell forecast
- the classifier-based breach probability

For the classifier warning flag, this project uses the Phase 5 recommended operating threshold:

`0.40`

This threshold balances strong recall with a manageable alert burden.

In [9]:
classifier_threshold = 0.40

dashboard_df["classifier_breach_probability"] = breach_prob
dashboard_df["classifier_warning_flag"] = (
    dashboard_df["classifier_breach_probability"] >= classifier_threshold
).astype(int)

dashboard_df[[
    "date",
    "terminal_id",
    "target_dwell_hours",
    "predicted_dwell",
    "risk_tier",
    "actual_breach_24h",
    "classifier_breach_probability",
    "classifier_warning_flag"
]].head()

,date,terminal_id,target_dwell_hours,predicted_dwell,risk_tier,actual_breach_24h,classifier_breach_probability,classifier_warning_flag
912,2024-07-01,T01,18.2,21.174817,Elevated Risk,0,0.753278,1
913,2024-07-02,T01,16.5,20.234585,Elevated Risk,0,0.545978,1
914,2024-07-03,T01,20.4,20.413917,Elevated Risk,0,0.362366,0
915,2024-07-04,T01,17.7,20.555955,Elevated Risk,0,0.442532,1
916,2024-07-05,T01,18.4,20.759380,Elevated Risk,0,0.507393,1


## 8. Create plain-English dashboard labels

This section adds short labels that will make the Streamlit dashboard easier to read. These labels are not new model outputs. They are presentation-friendly summaries based on the existing prediction and classification results.

In [10]:
def assign_dashboard_priority(row):
    if row["classifier_warning_flag"] == 1 and row["risk_tier"] in ["High Risk", "Threshold Breach Warning"]:
        return "Urgent Review"
    elif row["classifier_warning_flag"] == 1:
        return "Watchlist"
    elif row["risk_tier"] in ["High Risk", "Threshold Breach Warning"]:
        return "Planning Review"
    elif row["risk_tier"] == "Elevated Risk":
        return "Monitor"
    else:
        return "Normal"


def create_plain_english_summary(row):
    summary = (
        f"Terminal {row['terminal_id']} has predicted dwell of "
        f"{row['predicted_dwell']:.1f} hours and is classified as "
        f"{row['risk_tier']}."
    )

    if row["classifier_warning_flag"] == 1:
        summary += (
            f" The breach classifier estimates a "
            f"{row['classifier_breach_probability']:.1%} probability of a 24-hour breach."
        )

    if row["resource_pressure_flag"] == 1:
        summary += (
            " Resource pressure is present based on yard occupancy and/or crew availability."
        )

    return summary


dashboard_df["dashboard_priority"] = dashboard_df.apply(assign_dashboard_priority, axis=1)
dashboard_df["plain_english_summary"] = dashboard_df.apply(create_plain_english_summary, axis=1)

dashboard_df[[
    "date",
    "terminal_id",
    "predicted_dwell",
    "risk_tier",
    "classifier_breach_probability",
    "dashboard_priority",
    "plain_english_summary"
]].head()

,date,terminal_id,predicted_dwell,risk_tier,classifier_breach_probability,dashboard_priority,plain_english_summary
912,2024-07-01,T01,21.174817,Elevated Risk,0.753278,Watchlist,Terminal T01 has predicted dwell of 21.2 hours...
913,2024-07-02,T01,20.234585,Elevated Risk,0.545978,Watchlist,Terminal T01 has predicted dwell of 20.2 hours...
914,2024-07-03,T01,20.413917,Elevated Risk,0.362366,Monitor,Terminal T01 has predicted dwell of 20.4 hours...
915,2024-07-04,T01,20.555955,Elevated Risk,0.442532,Watchlist,Terminal T01 has predicted dwell of 20.6 hours...
916,2024-07-05,T01,20.759380,Elevated Risk,0.507393,Watchlist,Terminal T01 has predicted dwell of 20.8 hours...


## 9. Select and order dashboard columns

The final dashboard file should include only the fields needed for the Streamlit app. This keeps the app simpler and avoids clutter.

In [11]:
dashboard_cols = [
    "date",
    "terminal_id",
    "terminal_name",
    "region",
    "target_dwell_hours",
    "predicted_dwell",
    "prediction_error",
    "absolute_error",
    "risk_tier",
    "actual_breach_24h",
    "regression_breach_flag",
    "classifier_breach_probability",
    "classifier_warning_flag",
    "resource_pressure_flag",
    "dashboard_priority",
    "plain_english_summary",
    "inbound_train_count",
    "inbound_car_count",
    "cars_on_hand",
    "yard_occupancy_pct",
    "crew_starts_available",
    "locomotive_availability_pct",
    "is_weekend",
    "month",
]

dashboard_output = dashboard_df[dashboard_cols].copy()

dashboard_output["date"] = pd.to_datetime(dashboard_output["date"])

dashboard_output.head()

,date,terminal_id,terminal_name,region,target_dwell_hours,predicted_dwell,prediction_error,absolute_error,risk_tier,actual_breach_24h,regression_breach_flag,classifier_breach_probability,classifier_warning_flag,resource_pressure_flag,dashboard_priority,plain_english_summary,inbound_train_count,inbound_car_count,cars_on_hand,yard_occupancy_pct,crew_starts_available,locomotive_availability_pct,is_weekend,month
912,2024-07-01,T01,Barstow,West,18.2,21.174817,2.974817,2.974817,Elevated Risk,0,0,0.753278,1,0,Watchlist,Terminal T01 has predicted dwell of 21.2 hours...,6,125,90,10.0,22,82.6,0,7
913,2024-07-02,T01,Barstow,West,16.5,20.234585,3.734585,3.734585,Elevated Risk,0,0,0.545978,1,0,Watchlist,Terminal T01 has predicted dwell of 20.2 hours...,14,340,226,25.1,22,87.0,0,7
914,2024-07-03,T01,Barstow,West,20.4,20.413917,0.013917,0.013917,Elevated Risk,0,0,0.362366,0,0,Monitor,Terminal T01 has predicted dwell of 20.4 hours...,12,282,109,12.1,20,89.8,0,7
915,2024-07-04,T01,Barstow,West,17.7,20.555955,2.855955,2.855955,Elevated Risk,0,0,0.442532,1,0,Watchlist,Terminal T01 has predicted dwell of 20.6 hours...,12,224,90,10.0,15,88.8,0,7
916,2024-07-05,T01,Barstow,West,18.4,20.759380,2.359380,2.359380,Elevated Risk,0,0,0.507393,1,0,Watchlist,Terminal T01 has predicted dwell of 20.8 hours...,12,233,90,10.0,15,93.9,0,7


## 10. Validate the dashboard-ready dataset

Before saving the file, we check the row count, date range, risk-tier counts, and priority counts. This confirms that the dashboard file contains the expected test-period output.

In [12]:
print("Dashboard output shape:", dashboard_output.shape)
print("Date range:", dashboard_output["date"].min(), "to", dashboard_output["date"].max())
print("\nRisk tier counts:")
print(dashboard_output["risk_tier"].value_counts())

print("\nDashboard priority counts:")
print(dashboard_output["dashboard_priority"].value_counts())

print("\nClassifier warning count:")
print(dashboard_output["classifier_warning_flag"].sum())

print("\nActual breach count:")
print(dashboard_output["actual_breach_24h"].sum())

Dashboard output shape: (1464, 24)
Date range: 2024-07-01 00:00:00 to 2024-12-30 00:00:00

Risk tier counts:
risk_tier
Normal           743
Elevated Risk    685
High Risk         36
Name: count, dtype: int64

Dashboard priority counts:
dashboard_priority
Normal           731
Watchlist        617
Monitor           80
Urgent Review     36
Name: count, dtype: int64

Classifier warning count:
653

Actual breach count:
242


## 11. Save the dashboard dataset

The final step saves the dashboard-ready CSV to:

`data/processed/dashboard_forecasts.csv`

The `data/processed/` folder is generated locally and is not intended to be tracked in GitHub.

In [13]:
dashboard_output.to_csv(OUTPUT_PATH, index=False)

print("Saved dashboard dataset to:", OUTPUT_PATH)
print("File exists:", OUTPUT_PATH.exists())

Saved dashboard dataset to: ../data/processed/dashboard_forecasts.csv
File exists: True


## 12. Reload and verify the saved file

This final check confirms that the CSV can be read back successfully. This is important because the Streamlit dashboard will read this same file.

In [14]:
reloaded_dashboard = pd.read_csv(OUTPUT_PATH, parse_dates=["date"])

print("Reloaded shape:", reloaded_dashboard.shape)
reloaded_dashboard.head()

Reloaded shape: (1464, 24)


,date,terminal_id,terminal_name,region,target_dwell_hours,predicted_dwell,prediction_error,absolute_error,risk_tier,actual_breach_24h,regression_breach_flag,classifier_breach_probability,classifier_warning_flag,resource_pressure_flag,dashboard_priority,plain_english_summary,inbound_train_count,inbound_car_count,cars_on_hand,yard_occupancy_pct,crew_starts_available,locomotive_availability_pct,is_weekend,month
0,2024-07-01,T01,Barstow,West,18.2,21.174817,2.974817,2.974817,Elevated Risk,0,0,0.753278,1,0,Watchlist,Terminal T01 has predicted dwell of 21.2 hours...,6,125,90,10.0,22,82.6,0,7
1,2024-07-02,T01,Barstow,West,16.5,20.234585,3.734585,3.734585,Elevated Risk,0,0,0.545978,1,0,Watchlist,Terminal T01 has predicted dwell of 20.2 hours...,14,340,226,25.1,22,87.0,0,7
2,2024-07-03,T01,Barstow,West,20.4,20.413917,0.013917,0.013917,Elevated Risk,0,0,0.362366,0,0,Monitor,Terminal T01 has predicted dwell of 20.4 hours...,12,282,109,12.1,20,89.8,0,7
3,2024-07-04,T01,Barstow,West,17.7,20.555955,2.855955,2.855955,Elevated Risk,0,0,0.442532,1,0,Watchlist,Terminal T01 has predicted dwell of 20.6 hours...,12,224,90,10.0,15,88.8,0,7
4,2024-07-05,T01,Barstow,West,18.4,20.759380,2.359380,2.359380,Elevated Risk,0,0,0.507393,1,0,Watchlist,Terminal T01 has predicted dwell of 20.8 hours...,12,233,90,10.0,15,93.9,0,7


## 13. Conclusion and next step

This notebook created the dashboard-ready dataset for Phase 7. It combines outputs from the regression model, risk-tier logic, breach classifier, and dashboard presentation logic into one CSV.

The next step is to build the Streamlit dashboard in:

`app/app.py`

The dashboard will read:

`data/processed/dashboard_forecasts.csv`

and display terminal risk, dwell forecasts, breach warnings, priority levels, and plain-English operating summaries.